# HAT CHAIR Scores
This notebook tests CHAIR on the HAT data for Partial Abstention.

In [ ]:
import json
import os
import pickle
from chair import CHAIR
from tqdm.auto import tqdm

%load_ext autoreload
%autoreload 2

In [ ]:
# global chair
coco_path = "coco_annotations"
cached_instance = "chair.pkl"
chair_evaluator = None
if os.path.exists(cached_instance):
    chair_evaluator = pickle.load(open(cached_instance, "rb"))
    print(f"loaded evaluator from cache: {cached_instance}")
else:
    print("cache not setted or not exist yet, building from scratch...")
    chair_evaluator = CHAIR(coco_path)
    pickle.dump(chair_evaluator, open(cached_instance, "wb"))
    print(f"cached evaluator to: {cached_instance}")

In [ ]:
# load data
hat_file = "./input-data/hat-385_llava-blip-minigpt-lure2-aloha-complete.json"
with open(hat_file, "r") as f:
    hat_data = json.load(f)

print(json.dumps(hat_data[0], indent=2))

In [ ]:
def image_id_from_filename(filename):
    """Gets the image id from the MS COCO filename.

    Args:
        filename (str): The MS COCO filename. Ex: COCO_val2014_000000016903.jpg

    Returns:
        str: The image id. Ex: 16903
    """
    return str(int(filename.split(".")[0].split("_")[-1]))


def compute_chair_overall(chair_scored_list):
    """
    Compute CHAIR metrics for a list of CHAIR scores.
    """
    # variables to hold
    num_caps = 0.0
    num_hallucinated_caps = 0.0
    hallucinated_word_count = 0.0
    coco_word_count = 0.0

    num_recall_gt_objects = 0.0
    num_gt_objects = 0.0

    for chair_output in chair_scored_list:
        num_caps += 1
        coco_word_count += len(chair_output["mscoco_generated_words"])
        num_hallucinated_caps += (
            1 if len(chair_output["mscoco_hallucinated_words"]) > 0 else 0
        )
        hallucinated_word_count += len(chair_output["mscoco_hallucinated_words"])
        num_gt_objects += len(chair_output["mscoco_gt_words"])
        num_recall_gt_objects += len(chair_output["recall_gt_objects"])

    # compute overall metrics
    chair_s = num_hallucinated_caps / num_caps
    chair_i = hallucinated_word_count / coco_word_count
    # add
    recall = num_recall_gt_objects / num_gt_objects if num_gt_objects > 0 else 0

    return {
        "CHAIRs": chair_s,
        "CHAIRi": chair_i,
        "Recall": recall,
    }

In [ ]:
vlms = ["llava", "blip", "minigpt", "lure2"]
for image in tqdm(hat_data):
    for vlm in vlms:
        if vlm in image["captions"]:
            # get the chair output for reference caption
            atomics = [
                x["statement"] for x in image["captions"][vlm]["atomics"]["reference"]
            ]
            output = chair_evaluator.compute_chair(
                atomics,
                [image_id_from_filename(image["file_name"])] * len(atomics),
                compact_output=True,
            )
            for idx, chair_output in enumerate(output["sentences"]):
                chair_output.pop("image_id")
                chair_output.pop("caption")
                image["captions"][vlm]["atomics"]["reference"][idx]["chair"] = (
                    chair_output
                )

In [ ]:
# save data
with open(
    "./input-data/hat-385_llava-blip-minigpt-lure2_aloha-chair-complete.json", "w"
) as f:
    json.dump(hat_data, f, indent=2, ensure_ascii=False)